# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/GenAI/Advanced Topics in RAG/Knowledge Graphs with LightRAG

/content/drive/MyDrive/GenAI/Advanced Topics in RAG/Knowledge Graphs with LightRAG


In [3]:
!pip install lightrag-hku PyPDF2 pyvis -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 32.8 MB/s eta 0:00:00


In [4]:
# import libraries
import os
from IPython.display import Markdown, display
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete, openai_embed
from lightrag.kg.shared_storage import initialize_pipeline_status
import PyPDF2
from google.colab import userdata

In [5]:
# Set the OpenAI API key in the environment
api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = api_key

# Create a directory

In [9]:
# Set the name for the working directory
WORKING_DIR = "./working_dir_pdf"
WORKSPACE = "book_greeks"

if not os.path.exists(WORKING_DIR):
    os.makedirs(WORKING_DIR)

In [10]:
# Build the RAG CONFIGURATIONS
rag = LightRAG(
    working_dir = WORKING_DIR,
    workspace= WORKSPACE,
    embedding_func = openai_embed,
    llm_model_func = openai_complete,
    llm_model_name = "gpt-6-astra"
)

INFO: [book_greeks] Created new empty graph file: ./working_dir_pdf/book_greeks/graph_chunk_entity_relation.graphml
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=4, timeout=240
INFO:  - keyword: None/None, host=None, max_async=4, timeout=240
INFO:  - query: None/None, host=None, max_async=4, timeout=240
INFO:  - vlm: None/None, host=None, max_async=4, timeout=240


In [11]:
# Using Colab / Jupyter
import nest_asyncio
nest_asyncio.apply()
await rag.initialize_storages()
await initialize_pipeline_status()


INFO: [book_greeks] Process 1731 KV load full_docs with 0 records
INFO: [book_greeks] Process 1731 KV load text_chunks with 0 records
INFO: [book_greeks] Process 1731 KV load full_entities with 0 records
INFO: [book_greeks] Process 1731 KV load full_relations with 0 records
INFO: [book_greeks] Process 1731 KV load entity_chunks with 0 records
INFO: [book_greeks] Process 1731 KV load relation_chunks with 0 records
INFO: [book_greeks] Process 1731 KV load llm_response_cache with 0 records
INFO: [book_greeks] Process 1731 doc status load doc_status with 0 records


# Data Prep

In [12]:
# Open the PDF and transcribe the text
pdf_text = []
with open("Greek-gods-story.pdf", "rb") as file:
  reader = PyPDF2.PdfReader(file)
  for page in reader.pages:
    pdf_text.append(page.extract_text())
pdf_text

['Greek Gods Stories\nA Story About Zeus\nZeus was the god of the sky and ruler of the gods. \nHe was once travelling with his son, Hermes. Both men were disguised \nas ordinary men. On their travels, they stopped at many different \nhouses asking for shelter and food, but they were refused every time. \nZeus and his son came to a house belonging to a very poor, elderly \ncouple. Baucis and her husband, Philemon, offered the visitors a \nmeal and somewhere to rest for the night. As refreshment, the couple \noffered the men their only jug of wine. Later that evening, Baucis noticed that the jug had been \nrefilled so she turned to her husband and told him that the men must be gods. \nZeus punished those who had refused them food and shelter by causing a terrible flood. All the \nhomes in the valley were destroyed, except for the hut belonging to Baucis and Philemon. To \nrepay their kindness, Zeus transformed their little hut into a temple, where they lived out their \nlives as priestes

In [13]:
# Save the data
with open("greek_gods.txt", "w") as file:
  for line in pdf_text:
    file.write(line)

In [14]:
# Build the RAG
with open("greek_gods.txt", "r") as file:
  await rag.ainsert(file.read())

INFO: Processing 1 document(s)
INFO: Parsing (native): doc-52c8afbec9e3e7f946a74539d5a1c008
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-52c8afbec9e3e7f946a74539d5a1c008
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: doc-52c8afbec9e3e7f946a74539d5a1c008
INFO: extract LLM func: 4 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)
INFO:  == LLM cache == saving: default:extract:207a0183cd42d37dc6c8847944b25c6e
INFO:  == LLM cache == saving: default:extract:2809a04d6ae852e5ce566149b0ae0ed0
INFO: Chunk 1 of 3 extracted 4 Ent + 4 Rel doc-52c8afbec9e3e7f946a74539d5a1c008-chunk-002
INFO:  == LLM cache == saving: default:extract:af4821b6c2b8642971ef67a7592463ce
INFO:  == LLM cache == saving: default:extract:dcc15d52afe61e142e5fbc531a216860
INFO:  == LLM cache == saving: default:extract:e257f6af83caf36a055b674263b59873
INFO: Chunk 2 of 3 extracted 39 Ent + 56 Rel doc-52c8afbec9e3e7f946a74539d5a1c008-chunk-000
INF

# RAG



*   NAIVE
*   Local
*   Global



In [28]:
from lightrag.utils import setup_logger
setup_logger(logger_name="lightrag", level="WARNING")

In [48]:
# Test query
query = "What in Wine in the story"

In [18]:
# Function for LightRAG
async def perform_rag(query, mode):
  response = await rag.aquery(
      query = query,
      param = QueryParam(
          mode = mode,
          enable_rerank= False
      )
  )
  return Markdown(response)

In [49]:
# Performing NAIVE RAG
response = await rag.aquery(
    query = query,
    param = QueryParam(
        mode = "naive",
        enable_rerank= False
    )
)
Markdown(response)

In the stories, **wine is a drink**. Baucis and Philemon offer their only jug of wine to Zeus and Hermes as refreshment.

If you mean **“Who is the god of wine?”**, the answer is **Dionysus**.

### References
* [1] Greek Gods Stories

In [50]:
# Perform with local RAG
await perform_rag(query, "local")

If you mean **“Who is the god of wine in the story?”**, the answer is **Dionysus**.

The gods asked him to get Hephaestus drunk so that Hephaestus could be brought to Mount Olympus to free Hera from the golden throne.

### References
* [1] Greek Gods Stories

In [51]:
# Perform RAG with global
await perform_rag(query, "global")

**Dionysus is the god of wine** in the story.

The other gods asked him to get Hephaestus drunk. Hephaestus was then taken to Mount Olympus, where he freed Hera from the golden throne’s unbreakable chains.

### References
* [1] A Story About Hephaestus

In [31]:
# Perform RAG with hybrid
await perform_rag(query, "hybrid")

**Aphrodite is the Greek goddess of love**, described in the stories as very beautiful. She became the wife of Hephaestus. [1]

She also unintentionally helped cause the **Trojan War**. Aphrodite promised Paris the love of the world’s most beautiful woman, Helen, and he chose Aphrodite as the fairest goddess. When Paris and Helen fled to Troy, Helen’s husband, the king of Sparta, declared war. [1]

### References

* [1] Greek Gods Stories

# Graph Visualization

In [32]:
# import libraries
import networkx as nx
from pyvis.network import Network
import matplotlib.pyplot as plt
import random
from IPython.display import HTML

In [47]:
# Load entities
G = nx.read_graphml(f"{WORKING_DIR}/{WORKSPACE}/graph_chunk_entity_relation.graphml")

# Create the Network
net = Network(
    notebook = True,
    cdn_resources = "remote"
)
net.from_nx(G)

# Add colors and titles to nodes
for node in net.nodes:
  node['color'] = "#{:06x}".format(random.randint(0, 0xFFFFFF))
  if "description" in node:
    node["title"] = node["description"]

# Add the titles to the edges
for edge in net.edges:
  if "description" in edge:
    edge["title"] = edge["description"]


# Save the file
net.show("graph.html")
net.save_graph("graph.html")

# Display
HTML(filename='graph.html')

graph.html


In [55]:
# Load entities
G = nx.read_graphml(f"{WORKING_DIR}/{WORKSPACE}/graph_chunk_entity_relation.graphml")

# Isolate
target_node = "Paris"
subgraph_nodes = list(G.neighbors(target_node)) + [target_node]

# Create a subgraph
subgraph = G.subgraph(subgraph_nodes)

# Create the Network
net = Network(
    notebook = True,
    cdn_resources = "remote"
)
net.from_nx(subgraph)

# Add colors and titles to nodes
for node in net.nodes:
  node['color'] = "#{:06x}".format(random.randint(0, 0xFFFFFF))
  if "description" in node:
    node["title"] = node["description"]

# Add the titles to the edges
for edge in net.edges:
  if "description" in edge:
    edge["title"] = edge["description"]


# Save the file
net.show("graph.html")
net.save_graph("graph.html")

# Display
HTML(filename='graph.html')

graph.html


['Athena', 'Aphrodite', 'Hera', 'Helen', 'Troy', 'Paris']
